# Lab 10: Multi-Perspective Customer Segmentation

**Learning Objectives:**
1. Organize features into business-driven perspectives
2. Apply optimal K selection separately to perspectives
3. **Merge perspectives using HC on centroids**
4. Profile and characterize merged segments
5. Interpret feature importance via R² decomposition
6. Translate clusters into business personas

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN, KMeans, AgglomerativeClustering
from sklearn.base import clone
from sklearn.metrics import pairwise_distances
from scipy.cluster.hierarchy import dendrogram
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load preprocessed data
df = pd.read_csv('../data/data_preprocessed.csv')
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Splitting feature names into groups
# Remember which metric_features we decided to keep?
metric_features = ['income',
 'frq',
 'rcn',
 'clothes',
 'kitchen',
 'small_appliances',
 'toys',
 'house_keeping',
 'per_net_purchase',
 'spent_online']

non_metric_features = df.columns[df.columns.str.startswith('oh_')].tolist() 

unused_features = [i for i in df.columns if i not in (metric_features+non_metric_features) ]

print("Metric features:", metric_features)
print("Non-metric features:", non_metric_features)
print("Unused features:", unused_features)

# Before we proceed

- Preprocessing steps already done:
    - Dealt with inconsistencies (duplicates, dataypes)
    - Scaling
    - Univariate outlier removal
    - Dropped redudant features (`age`) 

- Important extra-steps:
    - Consider dropping/transforming the variable `rcn`. Why?
    - Multivariate outlier removal   

In [ ]:
metric_features.remove('rcn')  # Removing 'rcn' as discussed previously

In [ ]:
# Remove outliers using DBSCAN
# Based on the hyperparameters that will be found in lab10
dbscan = DBSCAN(eps=1.9, min_samples=20, n_jobs=4)
dbscan_labels = dbscan.fit_predict(df[metric_features])

# Check outlier count
outlier_count = Counter(dbscan_labels)
print(f"DBSCAN results: {outlier_count}")
print(f"Outliers detected: {outlier_count[-1]}")
print(f"Core customers: {outlier_count[0]}")

# Save outliers for later classification
df_out = df[dbscan_labels == -1].copy()

# Filter to core customers only
df = df[dbscan_labels != -1].copy()
print(f"\nWorking with {len(df):,} core customers")
print(f"Outliers saved for later: {len(df_out)}")

# Defining the perspectives

- **Multi-Perspective Segmentation:** Different stakeholders need different analytical views
    - **Marketing Team** wants behavioral segments (how customers shop)
    - **Product Team** wants preference segments (what customers buy)
    - **Executive Team** wants ONE unified segmentation strategy


In [ ]:
# Define feature perspectives

# Behavioral features (how customers shop)
demographic_features = [
    'income',
    'frq',
    'per_net_purchase',
    'spent_online'
]

# Product Preference features (what customers buy)
preference_features = [
    'clothes',
    'kitchen',
    'small_appliances',
    'toys',
    'house_keeping'
]

# Combined metric features (for final analysis)
metric_features = demographic_features + preference_features

print("Demographic features:", demographic_features)
print("Product Preference features:", preference_features)
print("Total metric features:", len(metric_features))

# Clustering for each perspective 

##### For each clustering perspective we need to define 

- Clustering algorithm 
- Number of clusters 

## Demographic perspective

In [ ]:
df_dem = df[demographic_features].copy()
df_dem

In [ ]:
kmeans = KMeans(
    init='k-means++',
    n_init=20,
    random_state=42
)


In [ ]:
hierarchical = AgglomerativeClustering(
    metric='euclidean'
)

In [ ]:
# Helper functions from Labs 09-12

def get_ss(df, feats):
    """
    Calculate the sum of squares (SS) for the given DataFrame.
    The sum of squares is computed as the sum of the variances of each column
    multiplied by the number of non-NA/null observations minus one.
    
    Parameters:
    df (pandas.DataFrame): The input DataFrame for which the sum of squares is to be calculated.
    feats (list of str): A list of feature column names to be used in the calculation.
    
    Returns:
    float: The sum of squares of the DataFrame.
    """
    df_ = df[feats]
    ss = np.sum(df_.var() * (df_.count() - 1))
    return ss


def get_ssb(df, feats, label_col):
    """
    Calculate the between-group sum of squares (SSB) for the given DataFrame.
    The between-group sum of squares is computed as the sum of the squared differences
    between the mean of each group and the overall mean, weighted by the number of observations
    in each group.
    
    Parameters:
    df (pandas.DataFrame): The input DataFrame containing the data.
    feats (list of str): A list of feature column names to be used in the calculation.
    label_col (str): The name of the column in the DataFrame that contains the group labels.
    
    Returns:
    float: The between-group sum of squares of the DataFrame.
    """
    ssb_i = 0
    for i in np.unique(df[label_col]):
        df_ = df.loc[:, feats]
        X_ = df_.values
        X_k = df_.loc[df[label_col] == i].values
        
        ssb_i += (X_k.shape[0] * (np.square(X_k.mean(axis=0) - X_.mean(axis=0))))
    
    ssb = np.sum(ssb_i)
    return ssb


def get_ssw(df, feats, label_col):
    """
    Calculate the sum of squared within-cluster distances (SSW) for a given DataFrame.
    
    Parameters:
    df (pandas.DataFrame): The input DataFrame containing the data.
    feats (list of str): A list of feature column names to be used in the calculation.
    label_col (str): The name of the column containing cluster labels.
    
    Returns:
    float: The sum of squared within-cluster distances (SSW).
    """
    feats_label = feats + [label_col]
    df_k = df[feats_label].groupby(by=label_col).apply(
        lambda col: get_ss(col, feats),
        include_groups=False
    )
    return df_k.sum()


def get_rsq(df, feats, label_col):
    """
    Calculate the R-squared value for a given DataFrame and features.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing the data.
    feats (list): A list of feature column names to be used in the calculation.
    label_col (str): The name of the column containing the labels or cluster assignments.
    
    Returns:
    float: The R-squared value, representing the proportion of variance explained by the clustering.
    """
    df_sst_ = get_ss(df, feats)  # get total sum of squares
    df_ssw_ = get_ssw(df, feats, label_col)  # get ss within
    df_ssb_ = df_sst_ - df_ssw_  # get ss between
    # r2 = ssb/sst
    return (df_ssb_ / df_sst_)


def get_r2_scores(df, feats, clusterer, min_k=1, max_k=9):
    """
    Loop over different values of k. To be used with sklearn clusterers.
    """
    r2_clust = {}
    for n in range(min_k, max_k):
        clust = clone(clusterer).set_params(n_clusters=n)
        labels = clust.fit_predict(df)
        df_concat = pd.concat([df,
                               pd.Series(labels, name='labels', index=df.index)], axis=1)
        r2_clust[n] = get_rsq(df_concat, feats, 'labels')
    return r2_clust

In [ ]:
r2_scores = {}

r2_scores['kmeans'] = get_r2_scores(df_dem, demographic_features, kmeans)

for linkage in ['complete', 'average', 'single', 'ward']:
    r2_scores[linkage] = get_r2_scores(
        df_dem,                 # data
        demographic_features,   # features of perspective
        # use HClust, changing the linkage at each iteration
        hierarchical.set_params(linkage=linkage) 
    )


In [ ]:
r2_scores_df = pd.DataFrame(r2_scores)
r2_scores_df

In [ ]:
# Visualizing the R² scores for each cluster solution on demographic variables
r2_scores_df.plot.line(figsize=(10,7))

plt.title("Demographic Variables:\nR² plot for various clustering methods\n", fontsize=21)
plt.legend(title="Cluster methods", title_fontsize=11)
plt.xlabel("Number of clusters", fontsize=13)
plt.ylabel("R² metric", fontsize=13)
plt.show()

## Preferences / product perspective

In [ ]:
# CLASS EXERCISE
# Apply the same procedure to the preference features

df_prf = None

In [ ]:
r2_scores_pref = {}
r2_scores_pref['kmeans'] = get_r2_scores(df_prf, preference_features, kmeans)

for linkage in ['complete', 'average', 'single', 'ward']:
    r2_scores_pref[linkage] = get_r2_scores(
        df_prf, preference_features, hierarchical.set_params(linkage=linkage)
    )

# Visualizing the R² scores for each cluster solution on product variables
pd.DataFrame(r2_scores_pref).plot.line(figsize=(10,7))

plt.title("Preference Variables:\nR2 plot for various clustering methods\n", fontsize=21)
plt.legend(title="Cluster methods", title_fontsize=11)
plt.xlabel("Number of clusters", fontsize=13)
plt.ylabel("R2 metric", fontsize=13)
plt.show()


# Final solutions 

In [ ]:
# CLASS EXERCISE
# Apply the final clustering solution to the demographic features

kmeans_demo = KMeans(
    n_clusters=None,
    init='k-means++',
    n_init=20,
    random_state=42
)

In [ ]:
df['demographic_cluster'] = None

In [ ]:
# CLASS EXERCISE
# Apply the final clustering solution to the preference features    

kmeans_pref = KMeans(
    n_clusters=3,
    init='k-means++',
    n_init=20,
    random_state=42
)

In [ ]:
df['product_cluster'] = None

In [ ]:
print("Clustering complete!")
print(f"Behavioral clusters: {df['demographic_cluster'].nunique()}")
print(f"Preference clusters: {df['product_cluster'].nunique()}")

#### How many combinations of clusters we would have ? 

In [ ]:
crosstab = pd.crosstab(df['demographic_cluster'], df['product_cluster'])
crosstab


#### Are these 12 clusters balanced ? 

#### Are we finished ? 

. 

. 

. 

.


# Merge clusters

#### Apply Hierarchical Clustering on cluster centroids

In [ ]:
# Calculate centroids for each combination
df_centroids = df.groupby(['demographic_cluster', 'product_cluster'])[metric_features].mean()
df_centroids

In [ ]:
print(f"\nCentroids shape: {df_centroids.shape}")

#### We will assume ward linkage 

#### But need to decide the number of final clusters 

In [ ]:
# Fit hierarchical clustering with distance tracking
hclust = AgglomerativeClustering(
    linkage='ward',
    metric='euclidean',
    distance_threshold=0,
    n_clusters=None
)
hclust_labels = hclust.fit_predict(df_centroids)

# Build linkage matrix for dendrogram
# (Adapted from scikit-learn documentation)
counts = np.zeros(hclust.children_.shape[0])
n_samples = len(hclust.labels_)

for i, merge in enumerate(hclust.children_):
    current_count = 0
    for child_idx in merge:
        if child_idx < n_samples:
            current_count += 1  # Leaf node
        else:
            current_count += counts[child_idx - n_samples]
    counts[i] = current_count

linkage_matrix = np.column_stack([
    hclust.children_,
    hclust.distances_,
    counts
]).astype(float)

print("Linkage matrix ready for dendrogram!")

In [ ]:
# Plot dendrogram with cutting options
fig = plt.figure(figsize=(12, 6))

# Potential cutting heights to consider
y_threshold = 2.3  # Can adjust this!

dendrogram(
    linkage_matrix,
    labels=df_centroids.index,
    color_threshold=y_threshold,
    above_threshold_color='k'
)

plt.axhline(y=y_threshold, color='r', linestyle='--', linewidth=2, label=f'Cut at {y_threshold}')

plt.title(f'Hierarchical Clustering on 12 Centroids\n(Ward Linkage)', fontsize=16)
plt.xlabel('Combination (behavior_label, product_label)')
plt.ylabel('Euclidean Distance')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### Apply the HC with *n* clusters on the centroids 

In [ ]:
n_final_clusters = None

hclust_final = AgglomerativeClustering(
    linkage='ward',
    n_clusters=n_final_clusters
)

hclust_final_labels = hclust_final.fit_predict(df_centroids)
hclust_final_labels

In [ ]:
# Add to centroids dataframe
df_centroids['merged_labels'] = hclust_final_labels
df_centroids

In [ ]:
cluster_mapper = df_centroids['merged_labels'].to_dict()
cluster_mapper  

In [ ]:
# Map merged labels back to all customers

df['merged_labels'] = df.apply(
    lambda row: cluster_mapper[(row['demographic_cluster'], row['product_cluster'])],
    axis=1
)
df


In [ ]:
print(df['merged_labels'].value_counts().sort_index())

### Are the merged clusters balanced ? 

In [ ]:
# Quick distinctiveness check (R²)
r2_merged = get_rsq(df, metric_features, 'merged_labels')
print(f"\nMerged clustering R²: {r2_merged:.4f}")

##### How could you compare different final K merged solutions ? 

# Profiling 

## Metric features 

In [ ]:
# Get centroids for ALL segments
all_segments = sorted(df['merged_labels'].unique())
df_all_centroids = df.groupby('merged_labels')[metric_features].mean()

# Create figure with 2 subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# LEFT PLOT: Parallel coordinates for all clusters
plt.sca(ax1)
pd.plotting.parallel_coordinates(
    df_all_centroids.reset_index(),
    'merged_labels',
    color=sns.color_palette('Dark2', n_colors=len(all_segments)),
    ax=ax1
)
ax1.axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=1.5)
ax1.set_title('Parallel Coordinates: All Segments', fontsize=14, fontweight='bold')
ax1.set_ylabel('Standardized Value', fontsize=12)
ax1.set_xlabel('Features', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax1.get_legend().remove()

# RIGHT PLOT: Cluster sizes (ordered by cluster number)
cluster_sizes = df['merged_labels'].value_counts().sort_index()
colors = sns.color_palette('Dark2', n_colors=len(all_segments))

ax2.bar(cluster_sizes.index.astype(str), cluster_sizes.values, color=colors)
ax2.set_xlabel('Number of Customers', fontsize=12)
ax2.set_ylabel('Cluster', fontsize=12)
ax2.set_title('Cluster Sizes', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

#### What other plot you can chose to profile the metric features ? 

In [ ]:
all_segments = sorted(df['merged_labels'].unique())
df_all_centroids = df.groupby('merged_labels')[metric_features].mean().T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(
    df_all_centroids,
    cmap='PiYG',
    annot=True,
    ax=ax1,
    center=0
)

ax1.set_title('Heatmap: All Segments', fontsize=14, fontweight='bold')
ax1.set_ylabel('Metric Features', fontsize=12)
ax1.set_xlabel('Clusters', fontsize=12)
ax1.grid(False)


cluster_sizes = df['merged_labels'].value_counts().sort_index()
#colors = sns.color_palette('Dark2', n_colors=len(all_segments))

ax2.bar(cluster_sizes.index.astype(str), cluster_sizes.values, )
ax2.set_xlabel('Number of Customers', fontsize=12)
ax2.set_ylabel('Cluster', fontsize=12)
ax2.set_title('Cluster Sizes', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## Non metric features 

In [ ]:
pd.crosstab(df['merged_labels'], df['education'], normalize='index').plot.bar(
    stacked=True,
    figsize=(10, 6), color=sns.color_palette("Dark2", n_colors=len(df['education'].unique()))
)
plt.title("Education Level Distribution by Merged Segment", fontsize=14)
plt.xlabel("Merged Segment")
plt.ylabel("Proportion")
plt.legend(title="Education Level", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
def plot_profiling_barplots(df, merged_labels_col, features, n_cols):
    
  # Group the data by merged labels and sum up counts for each feature
    grouped_data = df.groupby(merged_labels_col)[features].sum(numeric_only=True).reset_index()

    # Calculate the number of rows and columns dynamically
    n_features = len(features)
    n_cols = n_cols  # Use the provided number of columns
    n_rows = (n_features // n_cols) + (n_features % n_cols > 0)  # Ensures we cover all features
    
    # Create subplots 
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 5))  # Adjust size based on number of rows and columns
    axes = axes.flatten()  # Flatten the axes array for easier iteration

    # Create separate bar plots for each feature
    for i, feature in enumerate(features):
        sns.barplot(x=merged_labels_col, y=feature, data=grouped_data, ax=axes[i])

        # Customize each subplot
        axes[i].set_title(f'{feature} by {merged_labels_col}')  # Title of the plot
        axes[i].set_xlabel(merged_labels_col)  # X-axis label
        axes[i].set_ylabel('Counts')  # Y-axis label
        axes[i].tick_params(axis='x', rotation=90)  # Rotate x-axis labels for better readability

    # Hide any empty subplots (if features are fewer than the total available subplots)
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_profiling_barplots(df, 'merged_labels', ['oh_status_Married', 'oh_status_Single', 'oh_status_Together',
       'oh_status_Widow'], n_cols=4)

# Feature Importances 

### Using $R^2$

Do you still remember the formula for $R^2$ calculation ? 

What proportion of each feature's total variance is explained by the cluster assignments?

In [ ]:
def get_ss_variables(df):
    """Get the SS for each variable
    """
    ss_vars = df.var() * (df.count() - 1)
    return ss_vars

def r2_variables(df, labels):
    """Get the R² for each variable
    """
    sst_vars = get_ss_variables(df)
    ssw_vars = np.sum(df.groupby(labels).apply(get_ss_variables))
    return 1 - ssw_vars/sst_vars

In [ ]:
r2_feats = r2_variables(df[metric_features + ['merged_labels']], 'merged_labels').drop('merged_labels')
r2_feats

In [ ]:
r2_feats.sort_values().plot.barh(figsize=(10,6))
plt.title("Feature Importances by R²", fontsize=14)
plt.xlabel("Features", fontsize=12)
plt.ylabel("R² Value", fontsize=12)
plt.xticks(rotation=45, ha='right') 
plt.show()

##### So what are the features that most differentiate the segments ? 

### Decision Tree Classifier 

- Train a Decision Tree Classifier model over the cluster labels
    - Estimate the respective feature importance (Gini importance)
    - Apply the trained model to classify multivariate outliers  

# Homework 
- Apply two clustering perspectives to your project dataset (experiment several)